# چارچوب Microsoft Agent — Azure OpenAI (API پاسخ‌ها)

در این نمونه کد، شما از **چارچوب Microsoft Agent (MAF)** برای ایجاد یک عامل ساده با پشتیبانی **Azure OpenAI** استفاده خواهید کرد که با **API پاسخ‌ها** کار می‌کند.

> **یادداشت مهاجرت:** این نمونه قبلاً از Semantic Kernel با مدل‌های GitHub استفاده می‌کرد. اکنون به چارچوب Microsoft Agent منتقل شده است و مدل‌های GitHub (منسوخ‌شده، بازنشسته در ژوئیه ۲۰۲۶) با Azure OpenAI که از API پاسخ‌ها پشتیبانی می‌کند جایگزین شده‌اند. در MAF، `OpenAIChatClient` به نقطه پایدار `/openai/v1/` از Azure OpenAI اشاره دارد و به طور پیش‌فرض از API پاسخ‌ها استفاده می‌کند.

هدف این نمونه نشان دادن مراحلی است که بعداً در نمونه‌های کد اضافی هنگام پیاده‌سازی الگوهای مختلف عامل به کار گرفته خواهد شد.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## واردکردن بسته‌های موردنیاز پایتون


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## تعریف یک ابزار

در چارچوب Microsoft Agent، یک **ابزار** تابع ساده‌ای در پایتون است که با `@tool` تزئین شده و عامل می‌تواند آن را فراخوانی کند. در ادامه، ابزاری را تعریف می‌کنیم که یک مقصد تعطیلات تصادفی را برمی‌گرداند و از تکرار مقصد قبلی جلوگیری می‌کند.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ایجاد عامل

در اینجا، عامل به نام `TravelAgent` را ایجاد می‌کنیم.

در این مثال، از دستورالعمل‌های بسیار ساده استفاده می‌کنیم. آزادید این دستورالعمل‌ها را تغییر دهید تا ببینید رفتار عامل چگونه تغییر می‌کند.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## اجرای عامل

اکنون می‌توانیم عامل را اجرا کنیم. یک `AgentSession` ایجاد می‌کنیم تا عامل مکالمه را در طول نوبت‌ها به خاطر بسپارد، سپس دو `user_inputs` ارسال می‌کنیم. اولی درخواست یک سفر است؛ دومی می‌گوید کاربر از پیشنهاد خوشش نیامده و درخواست دیگری دارد — عامل از تاریخچه جلسه به‌علاوه ابزار `get_random_destination` برای پاسخ دادن استفاده می‌کند.

می‌توانید این پیام‌ها را تغییر دهید تا ببینید عامل چگونه واکنش‌های متفاوتی نشان می‌دهد. پاسخ‌ها به صورت **جریان ورودی** توکن به توکن هستند.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
